In [2]:
import pandas as pd
df = pd.read_csv("/content/GlobalWeatherRepository.csv")
print("Dataset Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nData Types:\n", df.dtypes)
print("\nSample Data:\n", df.head())
print("\nNumerical Summary:\n", df.describe())
print("\nCategorical Summary:\n", df.describe(include='object'))

Dataset Shape: (123356, 41)

Columns:
 Index(['country', 'location_name', 'latitude', 'longitude', 'timezone',
       'last_updated_epoch', 'last_updated', 'temperature_celsius',
       'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph',
       'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in',
       'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius',
       'feels_like_fahrenheit', 'visibility_km', 'visibility_miles',
       'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise',
       'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination'],
      dtype='object')

Data Types:
 country                          object
location_name                    object
latitude                        float64
longitude 

In [3]:
print("\nMissing Values Per Column:\n", df.isnull().sum())
missing_percent = (df.isnull().sum() / len(df)) * 100
print("\nMissing Value Percentage:\n", missing_percent)
print("\nDuplicate Rows:", df.duplicated().sum())
for col in df.select_dtypes(include=['int64','float64']).columns:
    print(f"\nChecking anomalies in {col}")
    print("Min:", df[col].min(), "Max:", df[col].max())
for col in df.columns:
    if "date" in col.lower():
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print("\nData Coverage:")
        print("Start Date:", df[col].min())
        print("End Date:", df[col].max())


Missing Values Per Column:
 country                         0
location_name                   0
latitude                        0
longitude                       0
timezone                        0
last_updated_epoch              0
last_updated                    0
temperature_celsius             0
temperature_fahrenheit          0
condition_text                  0
wind_mph                        0
wind_kph                        0
wind_degree                     0
wind_direction                  0
pressure_mb                     0
pressure_in                     0
precip_mm                       0
precip_in                       0
humidity                        0
cloud                           0
feels_like_celsius              0
feels_like_fahrenheit           0
visibility_km                   0
visibility_miles                0
uv_index                        0
gust_mph                        0
gust_kph                        0
air_quality_Carbon_Monoxide     0
air_quality_Ozone  

In [4]:
df = df.drop_duplicates()
threshold = 0.4 * len(df)
df = df.dropna(thresh=threshold, axis=1)
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
for col in numeric_cols:
    df = df[df[col] >= 0]
print("Missing Values After Cleaning:\n", df.isnull().sum())


Missing Values After Cleaning:
 country                         0
location_name                   0
latitude                        0
longitude                       0
timezone                        0
last_updated_epoch              0
last_updated                    0
temperature_celsius             0
temperature_fahrenheit          0
condition_text                  0
wind_mph                        0
wind_kph                        0
wind_degree                     0
wind_direction                  0
pressure_mb                     0
pressure_in                     0
precip_mm                       0
precip_in                       0
humidity                        0
cloud                           0
feels_like_celsius              0
feels_like_fahrenheit           0
visibility_km                   0
visibility_miles                0
uv_index                        0
gust_mph                        0
gust_kph                        0
air_quality_Carbon_Monoxide     0
air_quality_Ozon

In [13]:
unwanted_columns = [
    'temperature_fahrenheit',
    'wind_mph',
    'pressure_in',
    'precip_in',
    'feels_like_fahrenheit',
    'visibility_miles',
    'last_updated',
    'gust_mph',
    'last_updated_epoch',
    'wind_degree',
    'timezone',
    'moonrise',
    'moonset',
    'moon_phase',
    'moon_illumination',
    'wind_direction',
    'air_quality_Carbon_Monoxide',
    'air_quality_Ozone',
    'air_quality_Nitrogen_dioxide',
    'air_quality_Sulphur_dioxide',
    'air_quality_PM2.5',
    'air_quality_PM10',
    'air_quality_gb-defra-index'
]
df.drop(columns=[col for col in unwanted_columns if col in df.columns], inplace=True)

print("\nColumns After Removal:")
print(df.columns)


Columns After Removal:
Index(['country', 'location_name', 'latitude', 'longitude',
       'temperature_celsius', 'condition_text', 'wind_kph', 'pressure_mb',
       'precip_mm', 'humidity', 'cloud', 'feels_like_celsius', 'visibility_km',
       'uv_index', 'gust_kph', 'air_quality_us-epa-index', 'sunrise', 'sunset',
       'Year', 'Month'],
      dtype='object')


In [14]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
for col in df.columns:
    if "temp" in col.lower():
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = (df[col] - 32) * 5/9
for col in df.columns:
    if "wind" in col.lower():
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col] * 1.60934
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
print("Unit Conversion & Normalization Done Successfully")


Unit Conversion & Normalization Done Successfully


In [15]:
date_col = None
for col in df.columns:
    if "date" in col.lower():
        date_col = col
if date_col:
    df[date_col] = pd.to_datetime(df[date_col])
    df['Year'] = df[date_col].dt.year
    df['Month'] = df[date_col].dt.month
    monthly_avg = df.groupby(['Year','Month'])[numeric_cols].mean().reset_index()
    print("\nMonthly Average Data:")
    print(monthly_avg.head())
    monthly_avg.to_csv("GlobalWeather_Monthly_Average.csv", index=False)


In [17]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['year_month'] = df['date'].dt.to_period('M')

    monthly_avg = df.groupby('year_month')[num_cols].mean()

    print("\nMonthly Aggregated Data:")
    print(monthly_avg.head())

In [18]:
df.to_csv("GlobalWeatherRepository_Cleaned.csv", index=False)
print("Preprocessed Dataset Saved Successfully!")

Preprocessed Dataset Saved Successfully!


In [19]:
from google.colab import files
files.download("GlobalWeatherRepository_Cleaned.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>